# Logging my journey learning LLMs and PyTorch

It is long overdue to start learning PyTorch, and there is no better way to do it than by building a toy LLM. I've gone through a list of papers that helped me to understand how LLMs come to be, and I think it's a good idea to build a toy LLM to better connect the theory with practice.

# Setup PyTorch

In [1]:
import torch

In [2]:
print(f"PyTorch Version: {torch.__version__}")
if (torch.cuda.is_available()):
    device = torch.device("cuda")
    print(f"Using GPU, CUDA version: {torch.version.cuda}, Number of GPUs: {torch.cuda.device_count()}")
else:
    device = torch.device("cpu")

PyTorch Version: 2.10.0+cu128
Using GPU, CUDA version: 12.8, Number of GPUs: 1


# Get a working tokenizer
Let's use OpenAI's tiktoken tokenizer - alghough I am actually interested in building a minimum tokenizer from scratch later.

In [3]:
!git clone https://github.com/openai/tiktoken.git

fatal: destination path 'tiktoken' already exists and is not an empty directory.


In [4]:
import tiktoken

In [5]:
enc = tiktoken.get_encoding("r50k_base")

In [6]:
enc.encode("monkey d luffy")

[49572, 288, 300, 15352]

In [7]:
enc.decode(enc.encode("monkey d luffy"))

'monkey d luffy'

# Get the WebText dataset

In [8]:
from datasets import load_dataset

In [9]:
openwebtext = load_dataset("openwebtext")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:104: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
You are not authenticated with the Hugging Face Hub in this notebook.
If the error persists, please let us know by opening an issue on GitHub (https://github.com/huggingface/huggingface_hub/issues/new).
  warnings.warn(


Resolving data files:   0%|          | 0/80 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/80 [00:00<?, ?it/s]

Loading dataset shards:   0%|          | 0/80 [00:00<?, ?it/s]

In [10]:
openwebtext

DatasetDict({
    train: Dataset({
        features: ['text'],
        num_rows: 8013769
    })
})

In [11]:
openwebtext['train']

Dataset({
    features: ['text'],
    num_rows: 8013769
})

## Test tokenizing the training data

In [12]:
enc.encode(openwebtext['train'][42]['text'])

[5159,
 8305,
 370,
 868,
 510,
 1165,
 1903,
 290,
 1719,
 2761,
 25446,
 736,
 284,
 3993,
 743,
 423,
 257,
 4633,
 2928,
 319,
 262,
 2612,
 11,
 257,
 2050,
 2523,
 198,
 198,
 8061,
 508,
 423,
 5876,
 38193,
 572,
 284,
 3993,
 743,
 307,
 379,
 3220,
 2526,
 286,
 2612,
 5287,
 11,
 4837,
 910,
 13,
 198,
 198,
 464,
 2050,
 11,
 3199,
 287,
 262,
 3427,
 8894,
 4913,
 11,
 3940,
 517,
 621,
 2026,
 11,
 830,
 661,
 329,
 1367,
 812,
 13,
 198,
 198,
 29193,
 1043,
 883,
 508,
 6989,
 1811,
 12513,
 286,
 3595,
 3993,
 547,
 517,
 1884,
 284,
 1205,
 262,
 4006,
 11,
 287,
 543,
 262,
 2612,
 10143,
 284,
 8901,
 6105,
 13,
 198,
 198,
 38897,
 910,
 2252,
 2267,
 318,
 2622,
 284,
 766,
 611,
 257,
 3092,
 286,
 3993,
 5640,
 2612,
 5287,
 393,
 262,
 2792,
 318,
 517,
 3716,
 13,
 198,
 198,
 1,
 42332,
 867,
 286,
 262,
 1243,
 326,
 4646,
 262,
 2863,
 286,
 2612,
 5287,
 635,
 4646,
 47104,
 26,
 922,
 5496,
 11,
 5517,
 11,
 3463,
 2994,
 290,
 407,
 9216,
 1583,
 5045,
 

# Embedding layer

In [13]:
vocabulary_size = enc.n_vocab
d_model = 768

embedding_layer = torch.nn.Embedding(vocabulary_size, d_model)

Here we encode the phrase "monkey d luffy", which is then fed into the embedding_layer,
which produces d_model=768 embeddings, one 768 dimentional vector per token

In [14]:
embedding_layer(torch.LongTensor([enc.encode("monkey d luffy")]))

tensor([[[-0.6097, -0.0953,  1.3531,  ..., -1.0828,  1.1839,  0.4693],
         [-0.0739,  0.3105, -0.2027,  ..., -0.5149, -0.4843, -0.6747],
         [ 1.1782, -0.3085, -0.4106,  ...,  0.8634,  0.0973, -0.1760],
         [ 1.4910,  1.2252,  1.1049,  ...,  0.7528, -0.4912,  1.3185]]],
       grad_fn=<EmbeddingBackward0>)

In [15]:
embedding_layer.weight.shape

torch.Size([50257, 768])

Here we can see that the embedding layer is rather simple - it is a n_vocab x d_model matrix. So the embedding process is simply taking w[encoding].

## Positional Encoding

In [16]:
class LearnablePositionalEncoding(torch.nn.Module):
    def __init__(self, max_seq_len, d_model):
        super().__init__()
        self.pos_embeddings = torch.nn.Embedding(max_seq_len, d_model)
    def forward(self, x):
        # x is of shape (batch_size, seq_len, d_model)
        seq_len = x.size(1)

        position_ids = torch.arange(seq_len, dtype=torch.long, device=x.device)

        pos_enc = self.pos_embeddings(position_ids)
        return x + pos_enc


In [17]:
pos_embed_layer = LearnablePositionalEncoding(1024, d_model)

In [18]:
pos_embed_layer(embedding_layer(torch.LongTensor([enc.encode("monkey d luffy")])))

tensor([[[ 0.5480,  1.4881,  1.3152,  ..., -1.9352,  0.6646,  0.4440],
         [ 0.9257,  0.6944,  0.4519,  ..., -1.0091, -0.6509, -1.3701],
         [ 1.3583, -1.2847,  1.0583,  ...,  4.2241, -1.5553, -1.4901],
         [ 1.5153,  1.2714,  2.4634,  ..., -0.5578, -0.7508,  2.7747]]],
       grad_fn=<AddBackward0>)

# Decoder Only Transformer Layer

This is the key part of the transformer architecture. Each attention layer is made of multi-head attention, normalization, and position-wise feedforward.

In GPT-2, normalization is moved at the beginning of each layer.

In [19]:
embedding = embedding_layer(torch.LongTensor([enc.encode("monkey d luffy")]))

## Layer Normalization

https://arxiv.org/pdf/1607.06450

In [20]:
layer_norm = torch.nn.LayerNorm(d_model)

In [21]:
x = layer_norm(embedding)

In [22]:
x

tensor([[[-0.5725, -0.0688,  1.3495,  ..., -1.0358,  1.1838,  0.4840],
         [-0.0633,  0.3276, -0.1942,  ..., -0.5116, -0.4805, -0.6741],
         [ 1.2372, -0.3235, -0.4308,  ...,  0.9068,  0.1025, -0.1844],
         [ 1.4355,  1.1656,  1.0434,  ...,  0.6857, -0.5776,  1.2603]]],
       grad_fn=<NativeLayerNormBackward0>)

In [23]:
x.shape

torch.Size([1, 4, 768])

## Multi-Head Attention

In [24]:
n_head = 12

# project into Q, K, V
projection = torch.nn.Linear(d_model, d_model * 3)

In [25]:
projection(x).shape

torch.Size([1, 4, 2304])

In [26]:
proj = projection(x).view(1, -1, 12, d_model // n_head * 3).permute(0, 2, 1, 3)

In [27]:
q, k, v = proj.chunk(3, dim=3)

In [28]:
import math
atten = torch.matmul(q, k.transpose(-2, -1)) / math.sqrt(d_model)

In [29]:
mask = torch.tril(torch.ones_like(atten[0][0]))

In [30]:
atten = atten.masked_fill(mask == 0, -float('inf'))

In [31]:
atten

tensor([[[[ 0.0142,    -inf,    -inf,    -inf],
          [-0.0111, -0.0231,    -inf,    -inf],
          [-0.1347,  0.0878,  0.1012,    -inf],
          [-0.0956, -0.0218,  0.0288,  0.1964]],

         [[-0.0197,    -inf,    -inf,    -inf],
          [-0.0057,  0.0379,    -inf,    -inf],
          [-0.0242, -0.0806,  0.0839,    -inf],
          [-0.0846,  0.1213, -0.0497,  0.1032]],

         [[ 0.1550,    -inf,    -inf,    -inf],
          [ 0.0722, -0.0161,    -inf,    -inf],
          [ 0.0632,  0.0494,  0.0382,    -inf],
          [-0.0638,  0.0220,  0.0701, -0.1198]],

         [[-0.0218,    -inf,    -inf,    -inf],
          [ 0.1211,  0.0525,    -inf,    -inf],
          [ 0.0777,  0.0074,  0.0127,    -inf],
          [-0.0279, -0.0128,  0.0378, -0.0874]],

         [[-0.0212,    -inf,    -inf,    -inf],
          [-0.0967, -0.1822,    -inf,    -inf],
          [ 0.0038,  0.1484,  0.1216,    -inf],
          [-0.0978,  0.0647, -0.1072, -0.2957]],

         [[ 0.0482,    -inf,  

In [32]:
softmax = torch.nn.Softmax(dim=-1)

atten = softmax(atten)

In [33]:
atten

tensor([[[[1.0000, 0.0000, 0.0000, 0.0000],
          [0.5030, 0.4970, 0.0000, 0.0000],
          [0.2845, 0.3554, 0.3602, 0.0000],
          [0.2199, 0.2367, 0.2490, 0.2944]],

         [[1.0000, 0.0000, 0.0000, 0.0000],
          [0.4891, 0.5109, 0.0000, 0.0000],
          [0.3269, 0.3089, 0.3642, 0.0000],
          [0.2237, 0.2748, 0.2316, 0.2699]],

         [[1.0000, 0.0000, 0.0000, 0.0000],
          [0.5221, 0.4779, 0.0000, 0.0000],
          [0.3376, 0.3330, 0.3293, 0.0000],
          [0.2393, 0.2608, 0.2736, 0.2263]],

         [[1.0000, 0.0000, 0.0000, 0.0000],
          [0.5172, 0.4828, 0.0000, 0.0000],
          [0.3485, 0.3249, 0.3266, 0.0000],
          [0.2484, 0.2522, 0.2653, 0.2341]],

         [[1.0000, 0.0000, 0.0000, 0.0000],
          [0.5214, 0.4786, 0.0000, 0.0000],
          [0.3048, 0.3523, 0.3429, 0.0000],
          [0.2508, 0.2950, 0.2484, 0.2057]],

         [[1.0000, 0.0000, 0.0000, 0.0000],
          [0.5305, 0.4695, 0.0000, 0.0000],
          [0.2905, 0.3

In [34]:
torch.matmul(atten, v).transpose(1, 2).reshape(1, 4, d_model)

tensor([[[ 0.3025, -1.3879,  0.0880,  ...,  0.2237,  0.3752, -0.3010],
         [ 0.8233, -0.3795,  0.3293,  ...,  0.6207,  0.4280, -0.0535],
         [ 0.9007,  0.1902,  0.0930,  ...,  0.0535,  0.1475, -0.3472],
         [ 1.1966,  0.0647, -0.0192,  ..., -0.1369,  0.1198, -0.3953]]],
       grad_fn=<UnsafeViewBackward0>)

### Putting it together

In [35]:
import torch.nn.functional as F
class MultiHeadAttention(torch.nn.Module):
    def __init__(self, d_model, n_heads):
        super().__init__()
        self.d_model = d_model
        self.n_heads = n_heads
        # project into K Q V
        self.input_linear = torch.nn.Linear(d_model, d_model * 3)
    def forward(self, x):
        batch_size, sequence_length, _ = x.size()
        proj = self.input_linear(x).view(batch_size, sequence_length, self.n_heads, self.d_model // self.n_heads * 3).permute(0, 2, 1, 3)
        q, k, v = proj.chunk(3, dim=-1)

        atten = torch.matmul(q, k.transpose(-2, -1)) / math.sqrt(self.d_model)
        mask = torch.tril(torch.ones_like(atten[0][0]))
        atten.masked_fill(mask == 0, -float('inf'))
        atten = F.softmax(atten, dim=-1)

        return torch.matmul(atten, v).transpose(1, 2).reshape(batch_size, sequence_length, d_model)


In [36]:
multi_head_attention = MultiHeadAttention(d_model, n_head)
multi_head_attention(x)

tensor([[[ 0.2066,  0.1527, -0.5549,  ..., -0.0951,  0.0159,  0.1770],
         [ 0.2089,  0.1998, -0.5970,  ..., -0.1013,  0.0305,  0.1496],
         [ 0.2003,  0.2217, -0.5689,  ..., -0.0479,  0.0060,  0.2139],
         [ 0.2270,  0.1764, -0.5107,  ..., -0.0776,  0.0316,  0.1430]]],
       grad_fn=<UnsafeViewBackward0>)

## Position-wise Feed Forward

In [37]:
d_hidden = 3072

In [38]:
class DenseFeedForward(torch.nn.Module):
    def __init__(self, d_model, d_hidden):
        super().__init__()
        self.model = torch.nn.Sequential(
            torch.nn.Linear(d_model, d_hidden),
            torch.nn.ReLU(),
            torch.nn.Linear(d_hidden, d_model)
        )
    def forward(self, x):
        return self.model(x)

In [39]:
y = multi_head_attention(x)

In [40]:
dff = DenseFeedForward(d_model, d_hidden)

In [41]:
dff(y)

tensor([[[ 0.0117, -0.0957,  0.0547,  ...,  0.0068, -0.0218, -0.0682],
         [ 0.0190, -0.0915,  0.0449,  ...,  0.0076, -0.0180, -0.0714],
         [ 0.0257, -0.0982,  0.0564,  ..., -0.0040, -0.0214, -0.0662],
         [ 0.0281, -0.1113,  0.0553,  ..., -0.0031, -0.0190, -0.0665]]],
       grad_fn=<ViewBackward0>)

## The Full Transformer Layer

In [42]:
class TransformerDecoderOnly(torch.nn.Module):
    def __init__(self, d_model, d_hidden, n_head):
        super().__init__()
        self.layer_norm_0 = torch.nn.LayerNorm(d_model)
        self.multi_head_attention = MultiHeadAttention(d_model, n_head)
        self.layer_norm_1 = torch.nn.LayerNorm(d_model)
        self.dff = DenseFeedForward(d_model, d_hidden)
    def forward(self, x):
        # layer normalization first
        x = self.layer_norm_0(x)

        # multi-head attention and residual
        identity = x
        x = self.multi_head_attention(x)
        x = x + identity

        # layer normalization before dense feed forward
        x = self.layer_norm_1(x)

        # dense feed forward and residual
        identity = x
        x = self.dff(x)
        return x + identity

In [43]:
transformer_decoder_only = TransformerDecoderOnly(d_model, d_hidden, n_head)

transformer_decoder_only(embedding)

tensor([[[-0.5319,  0.1316,  1.1350,  ..., -0.8684,  1.5626,  0.6662],
         [-0.1330,  0.0995,  0.0700,  ..., -0.2121,  0.0642, -0.6999],
         [ 1.4492, -0.4561, -0.1684,  ...,  1.2649,  0.4339,  0.2557],
         [ 1.5404,  1.3903,  1.1990,  ...,  0.7471,  0.1166,  1.1443]]],
       grad_fn=<AddBackward0>)

# Output Layer

The output layer is rather simple. matrix multiply with the input embedding matrix, and soft max. We can then reverse the tokenization. 

In [44]:
torch.matmul(transformer_decoder_only(embedding), embedding_layer.weight.transpose(0, 1))

tensor([[[ 39.4479,  91.2051,  -6.3712,  ...,   3.7943,  33.9262,  16.7787],
         [  3.0791,  14.0035,   7.4868,  ...,  -3.7934, -13.9190, -36.5923],
         [-36.9327,  -8.3926,  -8.3561,  ...,  58.4200,  -6.9160,  56.2999],
         [-56.5484,  10.7033,  48.4754,  ...,  37.8723,  58.5866, -14.2468]]],
       grad_fn=<UnsafeViewBackward0>)

In [45]:
class Output(torch.nn.Module):
    def __init__(self, embedding_layer_weights):
        super().__init__()
        self.layer_norm = torch.nn.LayerNorm(embedding_layer_weights.shape[1])
        self.embedding_layer_weights = embedding_layer_weights
    
    def forward(self, x):
        # x is of shape (batch_size, seq_length, d_model)
        # self.embedding_layer_weights is of shape (vocab_size, d_model)
        x = self.layer_norm(x)
        return torch.matmul(x, self.embedding_layer_weights.transpose(0, 1))

# The full model

In [46]:
class ToyGPT(torch.nn.Module):
    def __init__(self, d_model, d_hidden, n_head, vocab_size, layers, max_seq_len):
        super().__init__()
        self.embedding_layer = torch.nn.Embedding(vocabulary_size, d_model)
        self.pos_encoding_layer = LearnablePositionalEncoding(max_seq_len, d_model)
        self.transformers = torch.nn.Sequential()
        for i in range (0, layers):
            self.transformers.append(TransformerDecoderOnly(d_model, d_hidden, n_head))
        self.output_layer = Output(self.embedding_layer.weight)
    
    def forward(self, x):
        x = self.embedding_layer(x)
        x = self.pos_encoding_layer(x)
        x = self.transformers(x)
        return self.output_layer(x)


# Training

In [47]:
batch_size = 48
max_seq_len = 1024

In [48]:
encoder = tiktoken.get_encoding('r50k_base')

In [49]:
encoder.special_tokens_set

{'<|endoftext|>'}

In [52]:
from datasets import Dataset

rows = openwebtext["train"].num_rows
bos_token = '<|endoftext|>'

def tokenize(examples):
    out = enc.encode(examples['text']+ bos_token, allowed_special={bos_token})
    return {'tokenized_text': out}

tokenized_data = openwebtext["train"].map(tokenize, remove_columns=openwebtext["train"].column_names, num_proc=32)


In [57]:
data = tokenized_data.to_iterable_dataset(num_shards=8)

In [54]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [ ]:
import gc

model = ToyGPT(d_model, d_hidden, n_head, enc.n_vocab, 12, max_seq_len)

encoding = []
model.to(device)

optimizer = torch.optim.SGD(model.parameters(), lr=0.0005)
tokens_per_batch = (batch_size + 1) * max_seq_len // 2

model = torch.compile(model)
model.train()

print_line_threshold = 0

for line in enumerate(data):
    encoding.extend([enc._special_tokens[bos_token]] + line[1]['tokenized_text'])
    if len(encoding) > tokens_per_batch:
        residual = encoding[tokens_per_batch:]
        encoding = encoding[0:tokens_per_batch + 1]
        batch = None
        target = None
        for i in range(0, batch_size):
            batch_row = torch.LongTensor(encoding[i * max_seq_len // 2 : i * max_seq_len // 2 + max_seq_len])
            batch_row = batch_row.to(device)
            target_row = torch.LongTensor(encoding[i * max_seq_len // 2 + 1 : i * max_seq_len // 2 + max_seq_len + 1])
            target_row = target_row.to(device)
            if batch is None:
                batch = batch_row
                target = target_row
            else:
                batch = torch.cat((batch, batch_row), dim=0)
                target = torch.cat((target, target_row), dim=0)

        batch = batch.to(device)
        batch = batch.reshape(batch_size, max_seq_len)
        target = target.to(device)
        target = target.reshape(batch_size, max_seq_len)

        optimizer.zero_grad()

        with torch.autocast(device_type='cuda', dtype=torch.bfloat16):
            out = model(batch)
            loss = F.cross_entropy(out.view(-1, enc.n_vocab), target.view(-1))

        loss.backward()
        optimizer.step()

        if (line[0] > print_line_threshold):
            print("{} / {} Loss: {}".format(line[0], rows, loss.item()))
            print_line_threshold = (line[0] + 1000) // 1000 * 1000

        encoding = residual

        gc.collect()
    

28 / 8013769 Loss: 252.71063232421875
1001 / 8013769 Loss: 64.3955078125
2016 / 8013769 Loss: 47.72172927856445
3002 / 8013769 Loss: 44.56999588012695
4011 / 8013769 Loss: 42.32395553588867
5024 / 8013769 Loss: 41.35980224609375
6004 / 8013769 Loss: 39.9724235534668
7011 / 8013769 Loss: 39.525875091552734
8020 / 8013769 Loss: 37.93269729614258
9027 / 8013769 Loss: 46.36484909057617
10014 / 8013769 Loss: 37.25653076171875
11006 / 8013769 Loss: 39.142330169677734
12004 / 8013769 Loss: 37.1536865234375
13003 / 8013769 Loss: 37.89497375488281
14018 / 8013769 Loss: 36.85776901245117
15027 / 8013769 Loss: 35.929935455322266
16001 / 8013769 Loss: 39.05356979370117
17011 / 8013769 Loss: 35.076168060302734
